In [0]:
from pyspark.sql import functions as F

# Feature table created earlier
feature_table = "workspace.ecommerce.train_dataset"

features_df = spark.table(feature_table)

print("Loaded ML feature dataset")
features_df.display(5)

In [0]:
import mlflow
import mlflow.spark

# Set experiment
mlflow.set_experiment("/Users/abhinavjajoo19@gmail.com/day7_mlflow_experiment")

# Search runs
runs = mlflow.search_runs(order_by=["metrics.AUC DESC"])

print("Total runs found:", len(runs))

if len(runs) == 0:
    raise Exception("No MLflow runs found. Run Day 7 training first.")

# Best run
best_run = runs.iloc[0]
run_id = best_run.run_id

print("Best Run ID:", run_id)
print("Best AUC:", best_run["metrics.AUC"])

# Model URI
model_uri = f"runs:/{run_id}/model"

# Load model with Unity Catalog temp directory
model = mlflow.spark.load_model(
    model_uri,
    dfs_tmpdir="/Volumes/workspace/ecommerce/silver_volume/mlflow_tmp"
)

print("✅ Model loaded successfully")

In [0]:
from pyspark.ml.feature import VectorAssembler

feature_table = "workspace.ecommerce.train_dataset"

df = spark.table(feature_table)

assembler = VectorAssembler(
    inputCols=[
        "total_events",
        "total_purchases",
        "total_spent",
        "avg_price",
        "unique_products"
    ],
    outputCol="features"
)

features_df = assembler.transform(df)

features_df.select("user_id", "features").display(5)

In [0]:
predictions = model.transform(features_df)

predictions.select(
    "user_id",
    "prediction",
    "probability"
).display(10)

In [0]:
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F

In [0]:
predictions_final = predictions.withColumn(
    "purchase_probability",
    vector_to_array("probability")[1]
).select(
    "user_id",
    "purchase_probability"
)

predictions_final.display(10)

In [0]:
gold_table = "workspace.ecommerce.final_predictions_gold"

predictions_final.write.mode("overwrite").saveAsTable(gold_table)

print("Final prediction table created successfully")

In [0]:
predictions_final.display(10)